In [1]:
!pip uninstall -y googletrans httpx
!pip install httpx datasets pandas deep-translator

Found existing installation: httpx 0.28.1
Uninstalling httpx-0.28.1:
  Successfully uninstalled httpx-0.28.1


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
import time

def main():
    print("1. Mengunduh dataset GoEmotions dari Hugging Face...")
    dataset = load_dataset("go_emotions")
    
    label_mapping = {
        'love': ['love', 'admiration', 'caring', 'desire'],
        'joy': ['joy', 'amusement', 'excitement', 'optimism', 'pride', 'relief', 'gratitude'],
        'anger': ['anger', 'annoyance', 'disgust'],
        'sadness': ['sadness', 'grief', 'remorse', 'disappointment'],
        'fear': ['fear', 'nervousness']
    }

    print("2. Memetakan label (Mapping)...")
    # PERBAIKAN: Gunakan fungsi terpisah untuk membuat kolom baru
    def add_mapped_emotion(example):
        example['mapped_emotion'] = None # Set default kosong
        labels = [dataset['train'].features['labels'].feature.names[i] for i, val in enumerate(example['labels']) if val == 1]
        
        for target_emotion, variations in label_mapping.items():
            if any(var in labels for var in variations):
                example['mapped_emotion'] = target_emotion
                break # Jika sudah ketemu 1 emosi, langsung berhenti mencari
        return example

    # Gunakan .map() untuk menerapkan perubahan (menambah kolom)
    mapped_ds = dataset['train'].map(add_mapped_emotion)
    
    # Ubah ke Pandas
    df_all = mapped_ds.to_pandas()

    print("3. Memfilter dan Menyeimbangkan dataset (1000 per kelas)...")
    # Buang semua baris yang 'mapped_emotion'-nya kosong (berarti bukan dari 5 kelas kita)
    df_filtered = df_all.dropna(subset=['mapped_emotion'])
    
    df_balanced = pd.DataFrame()
    for emotion in label_mapping.keys():
        # Ambil tepat 1000 data per emosi
        subset = df_filtered[df_filtered['mapped_emotion'] == emotion].head(1000)
        df_balanced = pd.concat([df_balanced, subset])
    
    # Acak urutan baris
    df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)
    
    print("4. Memulai proses translasi ke Bahasa Indonesia pakai deep-translator...")
    print("PENTING: Proses ini memakan waktu. Biarkan script berjalan.")
    
    translator = GoogleTranslator(source='auto', target='id')
    translated_texts = []
    
    for index, row in df_balanced.iterrows():
        original_text = row['text']
        try:
            res = translator.translate(original_text)
            translated_texts.append(res)
        except Exception as e:
            print(f"Error di baris {index}: {e}. Menggunakan teks asli.")
            translated_texts.append(original_text)
            time.sleep(2) 
            
        if (index + 1) % 500 == 0:
            print(f"Translasi: {index + 1} / 5000 selesai...")
            
    df_balanced['text_id'] = translated_texts

    print("5. Menyimpan hasil akhir...")
    df_final = df_balanced[['text_id', 'mapped_emotion']]
    df_final.columns = ['text', 'label'] 
    
    filename = "dataset_emosi_indo_final.csv"
    df_final.to_csv(filename, index=False)
    
    print(f"\nEKSEKUSI SELESAI. Dataset tersimpan di: {filename}")
    print("Komposisi kelas:")
    print(df_final['label'].value_counts())

if __name__ == "__main__":
    main()

1. Mengunduh dataset GoEmotions dari Hugging Face...
2. Memetakan label (Mapping)...


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

3. Memfilter dan Menyeimbangkan dataset (1000 per kelas)...
4. Memulai proses translasi ke Bahasa Indonesia pakai deep-translator...
PENTING: Proses ini memakan waktu. Biarkan script berjalan.
Translasi: 500 / 5000 selesai...
Translasi: 1000 / 5000 selesai...
5. Menyimpan hasil akhir...

EKSEKUSI SELESAI. Dataset tersimpan di: dataset_emosi_indo_final.csv
Komposisi kelas:
label
love    1000
joy       84
Name: count, dtype: int64


: 